In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

aws_access_key = os.environ['AWS_ACCESS_KEY_ID']
aws_secret_key = os.environ['AWS_SECRET_ACCESS_KEY']

In [2]:
import pyspark
from pyspark.sql import SparkSession

conf = (
    pyspark.SparkConf()
        .setAppName('app_name')
        .set('spark.jars.packages', 
             'org.apache.hadoop:hadoop-aws:3.3.4,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3,software.amazon.awssdk:bundle:2.17.178,software.amazon.awssdk:url-connection-client:2.17.178')
        .set('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
        .set('spark.sql.catalog.hdfs_catalog', 'org.apache.iceberg.spark.SparkCatalog')
        .set('spark.sql.catalog.hdfs_catalog.type', 'hadoop')
        .set('spark.sql.catalog.hdfs_catalog.warehouse', 's3a://diplakehouse-106124480608-ap-northeast-1-an/test_iceberg_book/')
        .set('spark.sql.catalog.hdfs_catalog.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
        .set('spark.hadoop.fs.s3a.access.key', os.environ['AWS_ACCESS_KEY_ID'])
        .set('spark.hadoop.fs.s3a.secret.key', os.environ['AWS_SECRET_ACCESS_KEY'])
        .set('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
        .set('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
        .set('spark.hadoop.fs.s3a.path.style.access', 'true')
)

spark = SparkSession.builder.config(conf=conf).getOrCreate()
print("spark running")

26/04/29 18:19:13 WARN Utils: Your hostname, onimas-ThinkPad-X13-Gen-2i resolves to a loopback address: 127.0.1.1; using 192.168.11.4 instead (on interface wlp9s0)
26/04/29 18:19:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/onimas/.pyenv/versions/3.11.12/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/onimas/.ivy2/cache
The jars for the packages stored in: /home/onimas/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
software.amazon.awssdk#bundle added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d5086a9f-f0e6-4028-b3c8-f6d98d01661a;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.3 in central
	found software.amazon.awssdk#bundle;2.17.178 in central
	found software.amazon.eventstream#eventstream;1.0.1 in central
	found software.amazon.awssdk#url-connection-client;2.17.178 in central
	found software.amazon.awssdk#utils;2.17.178 in central
	found org.reactiv

spark running


In [3]:
spark.sql("""
CREATE TABLE hdfs_catalog.customers (
    customer_id INT,
    first_name STRING,
    last_name STRING,
    email STRING,
    charges FLOAT,
    state STRING)
USING iceberg
PARTITIONED BY (state)
""")

26/04/18 22:32:34 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


DataFrame[]

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("charges", FloatType(), True),
    StructField("state", StringType(), True)
])

df = spark.createDataFrame([], schema)
df.writeTo("hdfs_catalog.customers_new").partitionedBy("state").create()

print("Table created successfully")

26/04/18 23:31:36 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".                
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Table created successfully


In [4]:
spark.sql("""
CREATE TABLE hdfs_catalog.high_value_customers
USING iceberg
PARTITIONED BY (state)
AS SELECT customer_id, first_name, last_name, state, charges
FROM hdfs_catalog.customers
WHERE charges > 100
""")

DataFrame[]

In [6]:
# df_ctas = spark.read.table("hdfs_catalog.customers")

# (
#     df_ctas.filter(df_ctas.charges > 1000)
#     .select("customer_id", "first_name", "last_name", "state", "charges")
#     .writeTo("hdfs_catalog.high_value_customers")
#     .partitionedBy("state")
#     .create()
# )

In [3]:
spark.sql("""
SELECT * FROM hdfs_catalog.customers
""")

26/04/23 23:16:45 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


DataFrame[customer_id: int, first_name: string, last_name: string, email: string, charges: float, state: string]

In [9]:
spark.sql("""
ALTER TABLE hdfs_catalog.customers
ADD COLUMN phone_number STRING
""")

DataFrame[]

In [10]:
spark.sql("""
SELECT * FROM hdfs_catalog.customers
""")

DataFrame[customer_id: int, first_name: string, last_name: string, email: string, charges: float, state: string, phone_number: string]

In [4]:
spark.sql("""
ALTER TABLE hdfs_catalog.customers
RENAME COLUMN charges TO total_spent
""")

DataFrame[]

In [5]:
spark.sql("""
ALTER TABLE hdfs_catalog.customers
DROP COLUMN phone_number
""")

DataFrame[]

In [6]:
spark.sql("""
ALTER TABLE hdfs_catalog.customers
ADD PARTITION FIELD bucket(16, customer_id)
""")

DataFrame[]

In [7]:
spark.sql("ALTER TABLE hdfs_catalog.customers CREATE BRANCH dev_branch")

DataFrame[]

In [10]:
spark.sql("""
    SELECT snapshot_id, committed_at, operation 
    FROM hdfs_catalog.customers.snapshots
    ORDER BY committed_at DESC
""").show()

+-------------------+--------------------+---------+
|        snapshot_id|        committed_at|operation|
+-------------------+--------------------+---------+
|2339874173708448584|2026-04-19 19:00:...|   append|
+-------------------+--------------------+---------+



In [11]:
spark.sql("""
ALTER TABLE hdfs_catalog.customers CREATE BRANCH audit_branch 
AS OF VERSION 2339874173708448584 
RETAIN 30 DAYS 
WITH SNAPSHOT RETENTION 3 SNAPSHOTS 2 DAYS
""")

DataFrame[]

In [13]:
spark.sql("""
INSERT INTO hdfs_catalog.customers VALUES
(1, 'John', 'Doe', 'john.doe@fakemail.co', 123.45, 'CA'),
(2, 'Jane', 'Smith', 'jane.smith@mockmail.org', 89.99, 'NY'),
(3, 'Alice', 'Johnson', 'alice.j@samplemail.net', 150.75, 'TX'),
(4, 'Bob', 'Brown', 'bob_brown@myemail.biz', 200.00, 'FL'),
(5, 'Eve', 'Davis', 'eve.davis@demoemail.com', 75.50, 'WA')
""")

DataFrame[]

In [14]:
spark.sql("""
ALTER TABLE hdfs_catalog.customers
CREATE TAG EOY_tag
""")

DataFrame[]

In [16]:
spark.sql("""
SELECT * FROM hdfs_catalog.customers
""").show()

+-----------+----------+---------+--------------------+-----------+-----+
|customer_id|first_name|last_name|               email|total_spent|state|
+-----------+----------+---------+--------------------+-----------+-----+
|          4|       Bob|    Brown|bob_brown@myemail...|      200.0|   FL|
|          1|      John|      Doe|john.doe@fakemail.co|     123.45|   CA|
|          5|       Eve|    Davis|eve.davis@demoema...|       75.5|   WA|
|          2|      Jane|    Smith|jane.smith@mockma...|      89.99|   NY|
|          3|     Alice|  Johnson|alice.j@samplemai...|     150.75|   TX|
+-----------+----------+---------+--------------------+-----------+-----+



In [17]:
spark.sql("""
CREATE TABLE hdfs_catalog.updates(
    customer_id INT,
    first_name STRING,
    last_name STRING,
    email STRING,
    charges FLOAT,
    state STRING
)
USING iceberg
""")

DataFrame[]

In [18]:
spark.sql("""
INSERT INTO hdfs_catalog.updates VALUES
    (1, 'John', 'Doe', 'john.doe@fakemail.co', 130.00, 'CA'), 
    (6, 'Chris', 'Evans', 'chris.evans@hollywood.com', 300.00, 'CA'),
    (7, 'Natasha', 'Romanoff', 'natasha.r@spyworld.com', 180.50, 'NY')
""")

DataFrame[]

In [20]:
spark.sql("""
ALTER TABLE hdfs_catalog.customers
RENAME COLUMN total_spent TO charges
""")

DataFrame[]

In [21]:
spark.sql("""
MERGE INTO hdfs_catalog.customers AS target
USING hdfs_catalog.updates AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *
""")

DataFrame[]

In [3]:
spark.sql("""
SELECT * FROM hdfs_catalog.customers
""").show()

26/04/29 18:19:32 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

+-----------+----------+---------+--------------------+-------+-----+
|customer_id|first_name|last_name|               email|charges|state|
+-----------+----------+---------+--------------------+-------+-----+
|          6|     Chris|    Evans|chris.evans@holly...|  300.0|   CA|
|          1|      John|      Doe|john.doe@fakemail.co|  130.0|   CA|
|          7|   Natasha| Romanoff|natasha.r@spyworl...|  180.5|   NY|
|          4|       Bob|    Brown|bob_brown@myemail...|  200.0|   FL|
|          5|       Eve|    Davis|eve.davis@demoema...|   75.5|   WA|
|          2|      Jane|    Smith|jane.smith@mockma...|  89.99|   NY|
|          3|     Alice|  Johnson|alice.j@samplemai...| 150.75|   TX|
+-----------+----------+---------+--------------------+-------+-----+

